<a href="https://colab.research.google.com/github/dan-the-man7/lab_4_/blob/main/lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
%pip install -q openai python-dotenv pandas matplotlib

import os, json, time, re, random

API_KEY = None
try:                                   # --- Google Colab (Secrets panel) ---
    from google.colab import userdata
    API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # --- Local (.env file) ---
    from dotenv import load_dotenv
    load_dotenv()
    API_KEY = os.environ.get("Groq_API_key")

assert API_KEY, (
    "No API key found. In Colab add a secret named GROQ_API_KEY (key icon, "
    "left sidebar) and switch on 'Notebook access'. Locally, create a .env file."
)
print("Key loaded:", API_KEY[:4] + "..." + API_KEY[-4:])   # never print the whole

# OpenAI-compatible client pointed at Groq
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready. Model:", MODEL)

Key loaded: gsk_...ZNuR
Client ready. Model: llama-3.3-70b-versatile


In [8]:
TOKEN_LOG = []          # (label, prompt_tokens, completion_tokens, total_tokens)


def ask_llm(user_prompt,
            system_prompt="You are a helpful assistant.",
            temperature=0.7,
            max_tokens=500,
            label="",
            retries=6):
    """Send one single-turn chat request and return the assistant's text."""
    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            u = response.usage
            TOKEN_LOG.append((label or "unlabelled",
                              u.prompt_tokens, u.completion_tokens, u.total_tokens))
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "rate" in str(e).lower() or "429" in str(e):
                wait = 2 ** attempt + random.random()
                print(f"   [rate limited — sleeping {wait:.1f}s]")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"ask_llm failed after {retries} attempts")


raw = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You explain finance simply, for a non-expert."},
        {"role": "user",   "content": "In two sentences, what is microfinance?"},
    ],
    temperature=0.7,
    max_tokens=150,
)

print("Answer:\n", raw.choices[0].message.content.strip())
print("\nRole of the reply:", raw.choices[0].message.role)
print("Finish reason     :", raw.choices[0].finish_reason)
print("\nUsage:")
print("  prompt_tokens     =", raw.usage.prompt_tokens)
print("  completion_tokens =", raw.usage.completion_tokens)
print("  total_tokens      =", raw.usage.total_tokens)

print("\nQuestion passed through ask_llm():\n",
      ask_llm("Name one risk a microfinance lender faces.",
              label="smoke_test", max_tokens=60))

Answer:
 Microfinance refers to the provision of small loans, savings, and other financial services to individuals or groups who lack access to traditional banking services, often in developing countries or low-income communities. By offering these services, microfinance institutions aim to empower people to start or expand their own businesses, improve their financial stability, and ultimately lift themselves out of poverty.

Role of the reply: assistant
Finish reason     : stop

Usage:
  prompt_tokens     = 55
  completion_tokens = 71
  total_tokens      = 126

Question passed through ask_llm():
 One risk a microfinance lender faces is default risk, which is the risk that borrowers will be unable to repay their loans, resulting in financial losses for the lender. This risk is particularly high in microfinance due to the small loan sizes and the lack of collateral from borrowers, who are often low-income


Section 1.1 Student Reasoning

A system role sets constraints or permissions that influence and govern the entire conversation. The user role contains the specific request for the LLM. A token is a chunk of text produced from a request of a set of words and sentences. Billing is based on token count because, the compute cost scales with the number of tokens processed. For instance, a 500-word prompt and a 5000-word prompt will not have the same token usage amount. The 5000-word request would require more tokens to serve, therefore culminating to a higher compute cost. The first call had a token count of 55 for the prompt_tokens

In [9]:
QUESTION = "Suggest a name for a savings product for market traders in Accra."
N_RUNS = 5

runs = {}
for temp in (0.0, 1.2):
  print(f"Running {N_RUNS} calls at temperature={temp} ...")
  runs[temp] = [
      ask_llm(QUESTION, temperature=temp, max_tokens=40, label=f"temp_{temp}")
      for _ in range(N_RUNS)
  ]

for temp, answers in runs.items():
  print("\n")
  print(f"Temperature = {temp}")
  print("\n")
  for i, a in enumerate(answers, 1):
    print(f"[{i}] {a}")

print(f"\n Distinct answers at temperature 0.0 : {len(set(runs[0.0]))} / {N_RUNS}")
print(f"\n Distinct answers at temperature 1.2 : {len(set(runs[1.2]))} / {N_RUNS}")

Running 5 calls at temperature=0.0 ...
Running 5 calls at temperature=1.2 ...


Temperature = 0.0


[1] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[2] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[3] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[4] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this
[5] Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this


Temperature = 1.2


[1] Here are a few suggestions for a savings product tailored to market traders in Accra:

1. **MakolaMmoa**: "Ma

Student Reflection 1.2

At temperature 0.0, there is only 1 distinct answer out of the 5 answers. The remaining answers repeat what they are saying. At temperature 1.2, all 5 answers are distinct, with 2 answers suggesting the "Makola Savings", but providing different explanations for the name choice. For this regime, Temperature 0 will be more appropriate because, extraction has a single correct answer, therefore any differences across runs would be classified as errors, rather than creativity.

In [10]:
## Section 2

LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
  "L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
           "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
  "L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
           "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
  "L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
           "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")


6 letters loaded.


In [11]:
## Version 1

SUMMARY_SYSTEM_V1 = "You are a helpful assistant."
SUMMARY_PROMPT_V1 = "Summarise this:\n\n{letter}"

## Version 2
SUMMARY_SYSTEM_V2 = """You are an assistant to a loan officer at a Ghanaian microfinance institution. You write short factual briefs that let the officer triage applications quickly.

Rules you must follow:
1. Use only facts stated in the letter. Never invent, infer, estimate, or improve on a number, a business detail or a personal circumstance.
2. If an important item (Loan amount, income, collateral/guarantor, repayment plan) is not stated, say explicitly that it is not stated.
3. Stay neutral. No sympathy, no encouragement, and no view on whether the loan should be granted.
4. Output 3-4 sentences of plain prose. No headings, no bullet points, no preamble such as "Here is a summary"."""

SUMMARY_PROMPT_V2 = """Summarize the loan application below in 3-4 sentences.

Cover, in this order:
(a) who the applicant is and what business they run,
(b) how much they are requesting and for what purpose,
(c) the financial position they claim (income/profit/savings) and the repayment they propose,
(d) what security is offered, and any material information that is missing.

LOAN APPLICATION

{letter}"""

def summarize(letter_text, version="v2"):
  if version == "v1":
    return ask_llm(SUMMARY_PROMPT_V1.format(letter=letter_text), SUMMARY_SYSTEM_V1, temperature=0.7, max_tokens=300, label="summary_v1")

  return ask_llm(SUMMARY_PROMPT_V2.format(letter=letter_text), SUMMARY_SYSTEM_V2, temperature=0.0, max_tokens=300, label="summary_v2")


## Side by side comparison of L001 and L006
for lid in ["L002", "L006"]:
  print(f"LETTER {lid}")
  print("\nORIGINAL LETTER")
  print(LETTERS [lid])
  print("\nV1 (naive prompt, temperature 0.7)")
  print(summarize (LETTERS[lid], "v1"))
  print("\nV2 (engineered prompt, temperature 0.0)")
  print(summarize (LETTERS[lid], "v2"))
  print()

## final version used by the rest of the lab

SUMMARY_SYSTEM = SUMMARY_SYSTEM_V2
SUMMARY_PROMPT = SUMMARY_PROMPT_V2


LETTER L002

ORIGINAL LETTER
Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.

V1 (naive prompt, temperature 0.7)
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects it to improve after the festive season. He has no collateral, but promises to repay the loan as soon as possible.

V2 (engineered prompt, temperature 0.0)
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan. He is requesting GHS 25,000 to repair his trotro engine and settle personal debts. His current financial position, including income, profit, or savings, is not